<!-- FULL: keep, DEMO: keep -->
<div style='display:flex; align-items:center; justify-content:space-between; border-bottom: 3px solid rgb(255,106,0); padding-bottom:1em; margin-bottom:1em'>
<div>
<span style='color:rgb(22,60,105); font-size:1.8em; font-weight:bold;'>Introduction to Deep Learning</span><br>
<span style='color:rgb(0,85,100); font-size:1.3em;'>Session 6 &mdash; 1: Tensors &amp; Autograd</span><br>
<span style='color:rgb(0,85,100); font-size:1.0em;'>Magda Gregorová &nbsp;·&nbsp; THWS &nbsp;·&nbsp; May 2026</span>
</div>
<img src='../../Common/Pics/thws-logo_vert_en_orange-rgb.png' style='height:80px;'/>
</div>

<!-- FULL: keep, DEMO: keep -->
<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>Forward and backward pass</strong>
</div>

**Forward pass**
$$
\mathbf{x} = \mathbf{h}^{(0)} \xrightarrow{\Theta^{(1)}} \mathbf{h}^{(1)} \xrightarrow{\Theta^{(2)}} \cdots \xrightarrow{\Theta^{(L)}} \mathbf{h}^{(L)} = \hat{y} \xrightarrow{\ell} \mathcal{L}
$$
**Backward pass**
$$
\nabla_{\boldsymbol{\theta}}\,\mathcal{L}(\boldsymbol{\theta}) = \frac{\partial \mathcal{L}}{\partial \boldsymbol{\theta}} \xleftarrow{\Theta^{(1)}} \frac{\partial \mathcal{L}}{\partial \mathbf{h}^{(1)}} \xleftarrow{\Theta^{(2)}} \cdots \xleftarrow{\Theta^{(L)}} \frac{\partial \mathcal{L}}{\partial \mathbf{h}^{(L)}} \xleftarrow{\ell} \mathcal{L}
$$

<!-- FULL: keep, DEMO: delete -->
<img src='../../Common/Pics/CompGraph_1.svg' style='max-width:70%; margin:1em auto; display:block;'/>

- **Input nodes** — fixed data - need no gradient
- **Parameter nodes** — learnable - need gradient
- **Compute nodes** — results of operations - local gradients

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import torch
import torch.nn as nn

<!-- FULL: keep, DEMO: keep -->
<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>Tensors</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
PyTorch's fundamental data structure — an $n$-dimensional array, like numpy but with GPU support and automatic differentiation.

Key attribute: `requires_grad` — tells PyTorch to track operations on this tensor so gradients can be computed.

https://docs.pytorch.org/docs/2.11/generated/torch.Tensor.requires_grad.html

In [ ]:
# ordinary tensor — no gradient tracking
x = torch.tensor([2.0, 3.0])
print(f'requires_grad: {x.requires_grad}')
print(f'grad_fn:       {x.grad_fn}')

In [ ]:
# tensor with gradient tracking
theta = torch.tensor([1.0, 0.5], requires_grad=True)
print(f'requires_grad: {theta.requires_grad}')
print(f'grad_fn:       {theta.grad_fn}')   # None — leaf node

<!-- FULL: keep, DEMO: delete -->
<div style='border-left: 4px solid rgb(0,85,100); padding: 0.3em 0.8em; background: rgb(235,245,247); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(0,85,100); font-size:1.05em;'>&#8644; Alternatives &mdash; creating leaf tensors with requires_grad</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
Three equivalent ways to create a trainable parameter tensor:

- `torch.tensor([...], requires_grad=True)` — from Python data
- `torch.zeros(...).requires_grad_(True)` — in-place flag
- `nn.Parameter(...)` — preferred inside `nn.Module` (covered later)


In [ ]:
theta_a = torch.tensor([1.0, 0.5], requires_grad=True)
theta_b = torch.zeros(2).requires_grad_(True)
theta_c = nn.Parameter(torch.tensor([1.0, 0.5]))

print(f'tensor + flag:          requires_grad={theta_a.requires_grad}, grad_fn={theta_a.grad_fn}')
print(f'zeros + requires_grad_: requires_grad={theta_b.requires_grad}, grad_fn={theta_b.grad_fn}')
print(f'parameter:              requires_grad={theta_c.requires_grad}, grad_fn={theta_c.grad_fn}')

<!-- FULL: keep, DEMO: delete -->
<div style='border-left: 4px solid rgb(179,27,27); padding: 0.3em 0.8em; background: rgb(255,240,240); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(179,27,27); font-size:1.05em;'>&#9888; Failure case &mdash; creating tensor from another tensor</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
Passing an existing tensor to `torch.tensor()` with `requires_grad=True` raises a `RuntimeError`. Use `.clone().detach().requires_grad_(True)` instead.

- **`.clone()`** — independent data (no shared storage); without it two tensors share the same memory and modifying one silently changes the other
- **`.detach()`** — independent from any existing graph; without it gradients would try to flow back through the source tensor's history


<!-- FULL: keep, DEMO: delete -->
Without `.clone()`, two tensors share the same memory. Modifying one silently changes the other.

https://docs.pytorch.org/docs/2.11/generated/torch.clone.html#torch.clone

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])
y = x          # no copy — shared storage
y[0] = 99.0
print(f'x after modifying y: {x}')   # x also changed!

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])   # reset
z = x.clone()               # independent copy
z[0] = 99.0
print(f'x after modifying z: {x}')   # x unchanged

<!-- FULL: keep, DEMO: delete -->
The actual error when creating from another tensor:

In [ ]:
x_existing = torch.randn(3)

# Wrong — raises RuntimeError
theta_bad = torch.tensor(x_existing, requires_grad=True)

In [ ]:
# Correct
theta_ok = x_existing.clone().detach().requires_grad_(True)
print(f'requires_grad: {theta_ok.requires_grad}')

<!-- FULL: keep, DEMO: delete -->
<div style='border-left: 4px solid rgb(179,27,27); padding: 0.3em 0.8em; background: rgb(255,240,240); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(179,27,27); font-size:1.05em;'>&#9888; Failure case &mdash; in-place operations on requires_grad tensors</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
In-place operations modify a tensor's data without creating a new graph node. This corrupts the graph PyTorch needs for backpropagation.

In [ ]:
theta = torch.tensor([1.0, 0.5], requires_grad=True)
z = theta * 2.0
theta += 1.0   # in-place — RuntimeError!

<!-- FULL: keep, DEMO: keep -->
<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>Dynamic Computation Graph</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
Every operation on a `requires_grad` tensor is recorded in a **dynamic computation graph**. Each result gets a `grad_fn` — a reference to the operation that created it.

- **Input nodes** — fixed data - need no gradient; `requires_grad=False`, leaf
- **Parameter nodes** — learnable - need gradient; `requires_grad=True`, leaf
- **Compute nodes** — results of operations - local gradients; have `grad_fn`, not leaf

<!-- FULL: keep, DEMO: delete -->
<img src='../../Common/Pics/CompGraph_1.svg' style='max-width:70%; margin:1em auto; display:block;'/>

In [ ]:
# same decomposition as Session 4
theta_1 = torch.tensor(1.0, requires_grad=True)  # parameter
theta_0 = torch.tensor(0.5, requires_grad=True)  # parameter
x = torch.tensor(2.0)  # input data
y = torch.tensor(3.0)  # target data

In [ ]:
# parameter nodes — requires_grad=True, grad_fn=None (leaf)
print('Parameters:')
for name, t in [('theta_1', theta_1), ('theta_0', theta_0)]:
    print(f'  {name}: requires_grad={t.requires_grad}, grad_fn={t.grad_fn}, is_leaf={t.is_leaf}')

In [ ]:
# input nodes — requires_grad=False, grad_fn=None (leaf)
print('Data:')
for name, t in [('x', x), ('y', y)]:
    print(f'  {name}: requires_grad={t.requires_grad}, grad_fn={t.grad_fn}, is_leaf={t.is_leaf}')
print('  -> no gradients computed for data')

<!-- FULL: keep, DEMO: delete -->
Forward pass builds the graph:

In [ ]:
#   (1) yhat = theta_1 * x + theta_0
#   (2) z    = yhat - y
#   (3) L    = z ** 2
yhat = theta_1 * x + theta_0
z    = yhat - y
L    = z**2

In [ ]:
# compute nodes — have grad_fn, not leaf
print('Compute nodes:')
for name, t in [('yhat', yhat), ('z', z), ('L', L)]:
    print(f'  {name} {t}, grad_fn={t.grad_fn.__class__.__name__}, is_leaf={t.is_leaf}')

In [ ]:
# each node records what feeds into it
print('L.grad_fn:', L.grad_fn)
print('L.grad_fn.next_functions:')
for fn in L.grad_fn.next_functions:
    print(f'  {fn}')

<!-- FULL: keep, DEMO: delete -->
<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>Detach — removing a tensor from the graph</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
`.detach()` cuts a tensor out of the computation graph. The detached tensor has no `grad_fn` — it is treated as a constant.

https://docs.pytorch.org/docs/2.11/generated/torch.Tensor.detach.html#torch.Tensor.detach

No gradient will flow through it. We will see the full consequence after introducing `.backward()`.

In [ ]:
print(f'yhat.grad_fn:          {yhat.grad_fn}')   # in graph

yhat_detached = yhat.detach()
print(f'yhat_detached.grad_fn: {yhat_detached.grad_fn}')  # None
print(f'same value:            {torch.allclose(yhat, yhat_detached)}')

<!-- FULL: keep, DEMO: keep -->
<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'><code>.backward()</code> and <code>.grad</code></strong>
</div>

<!-- FULL: keep, DEMO: delete -->
Calling `.backward()` on a scalar loss traverses the computation graph in reverse, computing $\partial L / \partial \theta$ for every leaf tensor with `requires_grad=True`. The result is stored in `.grad`.

In [ ]:
# fresh setup
theta_1 = torch.tensor(1.0, requires_grad=True)  # parameter
theta_0 = torch.tensor(0.5, requires_grad=True)  # parameter
x = torch.tensor(2.0)  # input data
y = torch.tensor(3.0)  # target data

In [ ]:
# forward pass
yhat = theta_1 * x + theta_0  # dyhat/dtheta_1 = x, dyhat/dtheta_0 = 1
z    = yhat - y  # dz/dyhat = 1
L    = z**2  # dL/dz = 2z
print(f'yhat={yhat}, z={z}, L={L}')
print(f'dyhat/dtheta_1={x}, dL/dz={2*z}, dL/dtheta1=2zx={2*z*x}, dL/dtheta0=2z={2*z}')

In [ ]:
# backward pass
L.backward()

In [ ]:
# gradients accumulated at parameter leaves
print(f'theta_1.grad: {theta_1.grad}')   # dL/d(theta_1)
print(f'theta_0.grad: {theta_0.grad}')   # dL/d(theta_0)

In [ ]:
# data and compute nodes have no .grad
print(f'x.grad: {x.grad}')   # None — no requires_grad
print(f'z.grad: {z.grad}')   # None — compute node, not leaf

<!-- FULL: keep, DEMO: delete -->
<div style='border-left: 4px solid rgb(179,27,27); padding: 0.3em 0.8em; background: rgb(255,240,240); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(179,27,27); font-size:1.05em;'>&#9888; Failure case &mdash; calling .backward() twice</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
PyTorch frees the computation graph after `.backward()` to save memory. Calling it again raises a `RuntimeError`. Pass `retain_graph=True` if you need to call it multiple times.

In [ ]:
# forward pass
yhat = theta_1 * x + theta_0  # dyhat/dtheta_1 = x, dyhat/dtheta_0 = 1
z    = yhat - y  # dz/dyhat = 1
L    = z**2  # dL/dz = 2z
print(f'yhat={yhat}, z={z}, L={L}')
print(f'dyhat/dtheta_1={x}, dL/dz={2*z}, dL/dtheta1=2zx={2*z*x}, dL/dtheta0=2z={2*z}')

L.backward()   # first call — fine
L.backward()   # second call — RuntimeError!

<!-- FULL: keep, DEMO: delete -->
<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>Detach revisited — consequence for gradients</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
Now that we understand `.backward()`, we can see what detach actually does: gradients do not flow through a detached tensor.

Practical use: **logging the loss** without keeping the graph alive. `.item()` detaches a scalar to a Python float.

In [ ]:
# fresh setup
theta_1 = torch.tensor(1.0, requires_grad=True)  # parameter
theta_0 = torch.tensor(0.5, requires_grad=True)  # parameter
x = torch.tensor(2.0)  # input data
y = torch.tensor(3.0)  # target data
yhat = theta_1 * x + theta_0  # dyhat/dtheta_1 = x, dyhat/dtheta_0 = 1

In [ ]:
# backward through original yhat — works
L = (yhat - y) ** 2
L.backward()
print(f'theta_1.grad via yhat:          {theta_1.grad}')

In [ ]:
# reset
theta_1 = torch.tensor(1.0, requires_grad=True)  # parameter
theta_0 = torch.tensor(0.5, requires_grad=True)  # parameter
x = torch.tensor(2.0)  # input data
y = torch.tensor(3.0)  # target data
yhat = theta_1 * x + theta_0  # dyhat/dtheta_1 = x, dyhat/dtheta_0 = 1
yhat_d = yhat.detach()

# backward through detached yhat — no gradient flows
L = (yhat_d - y) ** 2
L.backward()
print(f'theta_1.grad via yhat_detached: {theta_1.grad}')  # None

<!-- FULL: keep, DEMO: keep -->
<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>Why <code>.zero_grad()</code>?</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
PyTorch **accumulates** gradients — each `.backward()` call **adds** to `.grad`. Always zero gradients before each backward pass in a training loop.

In [ ]:
theta = torch.tensor([1.0, 0.5], requires_grad=True)
loss  = (theta[1] * 2.0 + theta[0]) ** 2
loss.backward()
print(f'grad: {theta.grad}')

In [ ]:
theta.grad.zero_()
print(f'after zero_():  {theta.grad}')

In [ ]:
# re-run backward to have a gradient, then set to None
loss = (theta[1] * 2.0 + theta[0]) ** 2
loss.backward()
theta.grad = None
print(f'after grad=None: {theta.grad}')

<!-- FULL: keep, DEMO: delete -->
<div style='border-left: 4px solid rgb(0,85,100); padding: 0.3em 0.8em; background: rgb(235,245,247); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(0,85,100); font-size:1.05em;'>&#8644; Alternatives &mdash; zeroing gradients</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
- `tensor.grad.zero_()` — zeros in-place
- `tensor.grad = None` — releases memory; slightly faster
- `optimizer.zero_grad()` — preferred in training loops (covered later)
- `optimizer.zero_grad(set_to_none=True)` — default in PyTorch ≥ 2.0


<!-- FULL: keep, DEMO: delete -->
<div style='border-left: 4px solid rgb(179,27,27); padding: 0.3em 0.8em; background: rgb(255,240,240); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(179,27,27); font-size:1.05em;'>&#9888; Failure case &mdash; in-place operations during update — why torch.no_grad()</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
The update step `theta -= lr * theta.grad` is itself an in-place operation. Without `torch.no_grad()`, PyTorch would try to record it in the graph, causing errors on the next backward pass.


In [ ]:
theta = torch.tensor(1.0, requires_grad=True)
L = (theta * 2.0 - 3.0) ** 2
L.backward()

# Wrong — in-place modification tracked by graph
theta -= 0.1 * theta.grad   # RuntimeError!

In [ ]:
theta = torch.tensor(1.0, requires_grad=True)
L = (theta * 2.0 - 3.0) ** 2
L.backward()

# Correct — update outside the graph
with torch.no_grad():
    theta -= 0.1 * theta.grad
print(f'theta after update: {theta}')

<!-- FULL: keep, DEMO: keep -->
<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'><code>torch.no_grad()</code></strong>
</div>

<!-- FULL: keep, DEMO: delete -->
During evaluation — computing validation loss, making predictions — gradients are not needed. `torch.no_grad()` disables graph construction, saving memory and computation.

In [ ]:
theta = torch.tensor([1.0, 0.5], requires_grad=True)

In [ ]:
# with grad tracking (default)
z = theta[1] * 2.0 + theta[0]
print(f'grad tracking on:  z.grad_fn = {z.grad_fn}')

In [ ]:
# without grad tracking
with torch.no_grad():
    z = theta[1] * 2.0 + theta[0]
print(f'grad tracking off: z.grad_fn = {z.grad_fn}')

<!-- FULL: keep, DEMO: keep -->
<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>Manual SGD training loop</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
The update rule from Session 5: $\theta \leftarrow \theta - \eta \, \nabla_{\theta} L$

Repeat until convergence
1. Forward pass
2. Compute loss
3. Get gradients
4. Update parameters

In [ ]:
# fresh setup
theta_1 = torch.tensor(1.0, requires_grad=True)  # parameter
theta_0 = torch.tensor(0.5, requires_grad=True)  # parameter
x = torch.tensor(2.0)  # input data
y = torch.tensor(3.0)  # target data

# forward pass
yhat = theta_1 * x + theta_0  # dyhat/dtheta_1 = x, dyhat/dtheta_0 = 1
L    = (yhat - y)**2  # dL/dyhat = 2(yhat-y)
print(f'yhat={yhat}, L={L}')
print(f'dyhat/dtheta_1={x}, dL/dyhat={2*(yhat-y)}, dL/dtheta1={2*(yhat-y)*x}, dL/dtheta0={2*(yhat-y)}')

In [ ]:
# SGD to optimize parameters

lr = 0.5
losses = []  #  montior losses
for step in range(3):

    # 1. forarward
    yhat = theta_1 * x + theta_0  # dyhat/dtheta_1 = x, dyhat/dtheta_0 = 1

    # 2. loss
    L = (yhat - y)**2  # dL/dyhat = 2(yhat-y)
    print('='*50)
    print(f'step {step} loss={L}')
    losses.append(L)

    print(f'step {step} theta_1 before backward: {theta_1}, requires_grad={theta_1.requires_grad}, grad_fn={theta_1.grad_fn}, grad={theta_1.grad}, is_leaf={theta_1.is_leaf}')

    # 3 — backward
    L.backward()

    print(f'step {step} theta_1 after backward: {theta_1}, requires_grad={theta_1.requires_grad}, grad_fn={theta_1.grad_fn}, grad={theta_1.grad}, is_leaf={theta_1.is_leaf}')

    # 4 — update
    # theta_1 = theta_1 - lr * theta_1.grad
    # theta_0 = theta_0 - lr * theta_0.grad

    # 4b — update (must be outside the graph)
    with torch.no_grad():
        theta_1 -= lr * theta_1.grad
        theta_0 -= lr * theta_0.grad
        dL_dtheta1 = 2*(yhat-y)*x

    print(f'step {step} theta_1 after update: {theta_1}, requires_grad={theta_1.requires_grad}, grad_fn={theta_1.grad_fn}, grad={theta_1.grad}, is_leaf={theta_1.is_leaf}')
    print(f'manual gradient theta_1: {dL_dtheta1}')

    # 5 — zero out gradients
    theta_1.grad.zero_()
    theta_0.grad.zero_()


print(f'Final loss: {losses[-1]:.4f}')
print(f'theta_1: {theta_1},  theta_0: {theta_0}')

<!-- FULL: keep, DEMO: keep -->
<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>Training on real data</strong>
</div>

In [ ]:
import matplotlib.pyplot as plt
import sys

# path to assignment 1
assignment_path = '../../Assignment_1/'
sys.path.append(assignment_path)
from helpers import load_data

X, y = load_data(assignment_path+'data.csv')

In [ ]:
# use first feature only for clarity
X_train = X[:, 1:2]

# parameters initialised to zero
theta_1 = torch.zeros(1, 1, requires_grad=True)
theta_0 = torch.zeros(1, 1, requires_grad=True)
lr      = 0.1

In [ ]:
losses = []
for epoch in range(50):

    # 1 & 2 — forward + loss
    yhat = X_train @ theta_1.T + theta_0
    L    = ((yhat - y) ** 2).mean()
    losses.append(L.item())

    # 3 — backward
    L.backward()

    # 4 — update (must be outside the graph)
    with torch.no_grad():
        theta_1 -= lr * theta_1.grad
        theta_0 -= lr * theta_0.grad

    # 5 — zero gradients
    theta_1.grad.zero_()
    theta_0.grad.zero_()

print(f'Final loss: {losses[-1]:.4f}')
print(f'theta_1: {theta_1.item():.4f},  theta_0: {theta_0.item():.4f}')

<!-- FULL: keep, DEMO: delete -->
<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>Loss curve</strong>
</div>

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(losses, color='#005564', linewidth=2)
ax.set_xlabel('epoch')
ax.set_ylabel('MSE loss')
ax.set_title('Training curve — manual SGD', color='#163C69', fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

<!-- FULL: keep, DEMO: delete -->
<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>Fit visualisation</strong>
</div>

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(X_train.numpy(), y.numpy(),
           alpha=0.3, s=15, color='#005564', label='data')

x_line = torch.linspace(X_train.min(), X_train.max(), 100).unsqueeze(1)
with torch.no_grad():
    y_line = x_line @ theta_1.T + theta_0
ax.plot(x_line.numpy(), y_line.numpy(),
        color='#FF6A00', linewidth=2, label='fit')

ax.set_xlabel('size (standardised)')
ax.set_ylabel('price (standardised)')
ax.set_title('Linear fit after training', color='#163C69', fontweight='bold')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

<!-- FULL: keep, DEMO: delete -->
This loop works but is limited: full-batch only, manual parameter management, and hard-coded SGD. `nn.Module`, `DataLoader`, and `torch.optim` solve all of these — covered in the next notebooks.

Further reading: https://docs.pytorch.org/docs/stable/notes/autograd.html
